# evaluation

## Building the graph

![Sports event reservation chatbot orchestration](architecture.svg)

In [1]:
# first, we start to define a versatile llm agent
from langgraph.graph import MessagesState
from langchain_ollama import OllamaLLM

MODEL_NAME = "qwen3.5:0.8b"

def ollama_llm(model_name=MODEL_NAME, temperature=0.6):
    return OllamaLLM(
        model=model_name,
        temperature=temperature, # temperature is not static since we need to be able to change it depending on the use-case (i.e. 0.0 for the router)
        reasoning=False # the model we chose is a reasoning model but we don't want the output to take time, so we disable it.
    ) 

In [ ]:
import json
from jinja2 import Environment, FileSystemLoader
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command

prompts = Environment(loader=FileSystemLoader("prompts"))
VALID_ROUTES = ("ticket_reservation", "question_answering")

def assistant_update(content: str) -> dict:
    return {"messages": [{"role": "assistant", "content": content}]}

# defining the nodes
def router(state: MessagesState):
    user_message = state['messages'][0].content

    llm = ollama_llm(temperature=0) # the router needs to be deterministic
    prompt = prompts.get_template("router.j2").render(user_message=user_message)
    llm_response = llm.invoke(prompt)

    route = json.loads(llm_response)["route"]
    if route not in VALID_ROUTES:
        raise ValueError(f"Unknown route: {route}")
    return Command(goto=route, update=assistant_update(llm_response))

def ticket_reservation(state: MessagesState):
    user_message = state['messages'][0].content
    llm = ollama_llm()
    prompt = prompts.get_template("ticket_reservation.j2").render(user_message=user_message)
    return assistant_update(llm.invoke(prompt))

def question_answering(state: MessagesState):
    user_message = state['messages'][0].content
    llm = ollama_llm()
    prompt = prompts.get_template("question_answering.j2").render(user_message=user_message)
    return assistant_update(llm.invoke(prompt))


# defining the whole graph
graph = StateGraph(MessagesState)
graph.add_node(router, destinations=VALID_ROUTES)
graph.add_node(ticket_reservation)
graph.add_node(question_answering)

graph.add_edge(START, "router")
graph.add_edge("ticket_reservation", END)
graph.add_edge("question_answering", END)

graph = graph.compile()

# testing the graph once
graph.invoke({
    "messages": [
        {
            'role': 'user',
            'content': 'What is the price for the game of the Eagles next Sunday ?'
        }
    ]
})


[router]
state {'messages': [HumanMessage(content='What is the price for the game of the Eagles next Sunday ?', additional_kwargs={}, response_metadata={}, id='da5cde38-28b1-4b09-ab20-d014fa7bd4e0')]}
User message: What is the price for the game of the Eagles next Sunday ?
llm_response {"route": "ticket_reservation"}
[ticket_reservation]


{'messages': [HumanMessage(content='What is the price for the game of the Eagles next Sunday ?', additional_kwargs={}, response_metadata={}, id='da5cde38-28b1-4b09-ab20-d014fa7bd4e0'),
  AIMessage(content='{"route": "ticket_reservation"}', additional_kwargs={}, response_metadata={}, id='77eb647d-dde8-48d2-babf-48e257e3fcd7', tool_calls=[], invalid_tool_calls=[]),
  AIMessage(content='{"response": "I am unable to provide real-time pricing information for sports events as I do not have access to current market data or live schedules. To find out the exact price, please check our official website or call our customer service team for up-to-date information."}', additional_kwargs={}, response_metadata={}, id='e22978f0-65b0-4539-9c1e-a37eddfc940f', tool_calls=[], invalid_tool_calls=[])]}

## Evaluation

Evaluation lies in 3 steps:

1. Evaluating the routing accuracy

2. Evaluating the trajectory taken by the graph

3. Evaluating the accuracy of the model in answering the right question

For each step, we will be conducting an experiment using a small toy gold dataset.

> Note : we could use web-based tools like Langgraph Studio and Langsmith or even Langfuse (and more...) to submit evaluation jobs and follow the results on a nice UI but it is not the point of this notebook. I also think that the less the better and thus that a notebook is perfectly fine for this task.

### Routing accuracy

Gold labels are the subgraph the router should choose for each user request:

- `ticket_reservation`: the user wants to book, buy, change, or cancel tickets
- `question_answering`: the user wants information (schedule, price, venue, policy, etc.)

In [ ]:
routing_dataset = [
    {
        "id": "route_001",
        "messages": [{"role": "user", "content": "I want to book 2 tickets for the Eagles game next Sunday."}],
        "route": "ticket_reservation",
    },
    {
        "id": "route_002",
        "messages": [{"role": "user", "content": "Reserve 4 seats for Yankees vs Red Sox on Saturday."}],
        "route": "ticket_reservation",
    },
    {
        "id": "route_003",
        "messages": [{"role": "user", "content": "Can you buy me a ticket for the Lakers game on Friday?"}],
        "route": "ticket_reservation",
    },
    {
        "id": "route_004",
        "messages": [{"role": "user", "content": "Please cancel my reservation for the Warriors game."}],
        "route": "ticket_reservation",
    },
    {
        "id": "route_005",
        "messages": [{"role": "user", "content": "Change my seats to section 112 for the Eagles game."}],
        "route": "ticket_reservation",
    },
    {
        "id": "route_006",
        "messages": [{"role": "user", "content": "What time does the Eagles game start next Sunday?"}],
        "route": "question_answering",
    },
    {
        "id": "route_007",
        "messages": [{"role": "user", "content": "Where is the Yankees vs Red Sox game being played?"}],
        "route": "question_answering",
    },
    {
        "id": "route_008",
        "messages": [{"role": "user", "content": "What's the bag policy at Lincoln Financial Field?"}],
        "route": "question_answering",
    },
    {
        "id": "route_009",
        "messages": [{"role": "user", "content": "What is the price for the game of the Eagles next Sunday?"}],
        "route": "question_answering",
    },
    {
        "id": "route_010",
        "messages": [{"role": "user", "content": "What's the refund policy if I can't attend?"}],
        "route": "question_answering",
    },
]

In [5]:
# TODO: perform the evaluation

### Trajectory optimization

Gold labels are the optimal node sequence through the graph.
In this orchestration that is always `router` followed by the correct specialist, then `END`.

In [3]:
trajectory_dataset = [
    {
        "id": "traj_001",
        "messages": [{"role": "user", "content": "I want to book 2 tickets for the Eagles game next Sunday."}],
        "trajectory": ["router", "ticket_reservation"],
    },
    {
        "id": "traj_002",
        "messages": [{"role": "user", "content": "Reserve 4 seats for Yankees vs Red Sox on Saturday."}],
        "trajectory": ["router", "ticket_reservation"],
    },
    {
        "id": "traj_003",
        "messages": [{"role": "user", "content": "Can you buy me a ticket for the Lakers game on Friday?"}],
        "trajectory": ["router", "ticket_reservation"],
    },
    {
        "id": "traj_004",
        "messages": [{"role": "user", "content": "Please cancel my reservation for the Warriors game."}],
        "trajectory": ["router", "ticket_reservation"],
    },
    {
        "id": "traj_005",
        "messages": [{"role": "user", "content": "Change my seats to section 112 for the Eagles game."}],
        "trajectory": ["router", "ticket_reservation"],
    },
    {
        "id": "traj_006",
        "messages": [{"role": "user", "content": "What time does the Eagles game start next Sunday?"}],
        "trajectory": ["router", "question_answering"],
    },
    {
        "id": "traj_007",
        "messages": [{"role": "user", "content": "Where is the Yankees vs Red Sox game being played?"}],
        "trajectory": ["router", "question_answering"],
    },
    {
        "id": "traj_008",
        "messages": [{"role": "user", "content": "What's the bag policy at Lincoln Financial Field?"}],
        "trajectory": ["router", "question_answering"],
    },
    {
        "id": "traj_009",
        "messages": [{"role": "user", "content": "What is the price for the game of the Eagles next Sunday?"}],
        "trajectory": ["router", "question_answering"],
    },
    {
        "id": "traj_010",
        "messages": [{"role": "user", "content": "What's the refund policy if I can't attend?"}],
        "trajectory": ["router", "question_answering"],
    },
]

In [6]:
# TODO: perform the evaluation

### Question answering accuracy

Gold labels are grounded in a small event catalog. `answer` is the canonical reply; `key_points` are the facts a correct answer must include.

In [4]:
EVENTS = {
    "eagles_cowboys": {
        "home": "Philadelphia Eagles",
        "away": "Dallas Cowboys",
        "date": "Sunday, August 23, 2026",
        "time": "1:00 PM ET",
        "venue": "Lincoln Financial Field",
        "starting_price_usd": 85,
    },
    "yankees_red_sox": {
        "home": "New York Yankees",
        "away": "Boston Red Sox",
        "date": "Saturday, August 22, 2026",
        "time": "7:05 PM ET",
        "venue": "Yankee Stadium",
        "starting_price_usd": 45,
    },
    "lakers_warriors": {
        "home": "Los Angeles Lakers",
        "away": "Golden State Warriors",
        "date": "Friday, August 21, 2026",
        "time": "7:30 PM PT",
        "venue": "Crypto.com Arena",
        "starting_price_usd": 120,
    },
}

POLICIES = {
    "bag": "Clear bags only, maximum 12 x 6 x 12 inches.",
    "parking": "On-site parking is $40 per vehicle.",
    "children": "Children under 2 years old do not need a ticket.",
    "refunds": "Full refunds are available up to 24 hours before kickoff.",
}

qa_dataset = [
    {
        "id": "qa_001",
        "messages": [{"role": "user", "content": "What time does the Eagles game start next Sunday?"}],
        "answer": "The Philadelphia Eagles vs Dallas Cowboys game on Sunday, August 23, 2026 kicks off at 1:00 PM ET at Lincoln Financial Field.",
        "key_points": ["1:00 PM ET", "Dallas Cowboys", "Lincoln Financial Field"],
    },
    {
        "id": "qa_002",
        "messages": [{"role": "user", "content": "Where is the Yankees vs Red Sox game being played?"}],
        "answer": "The New York Yankees vs Boston Red Sox game on Saturday, August 22, 2026 is at Yankee Stadium.",
        "key_points": ["Yankee Stadium"],
    },
    {
        "id": "qa_003",
        "messages": [{"role": "user", "content": "How much do Lakers tickets start at for Friday's game?"}],
        "answer": "Tickets for the Los Angeles Lakers vs Golden State Warriors game on Friday, August 21, 2026 start at $120.",
        "key_points": ["$120", "Golden State Warriors"],
    },
    {
        "id": "qa_004",
        "messages": [{"role": "user", "content": "What is the price for the game of the Eagles next Sunday?"}],
        "answer": "Tickets for the Philadelphia Eagles vs Dallas Cowboys game on Sunday, August 23, 2026 start at $85.",
        "key_points": ["$85"],
    },
    {
        "id": "qa_005",
        "messages": [{"role": "user", "content": "What's the bag policy at Lincoln Financial Field?"}],
        "answer": "Clear bags only, maximum 12 x 6 x 12 inches.",
        "key_points": ["clear bags", "12 x 6 x 12"],
    },
    {
        "id": "qa_006",
        "messages": [{"role": "user", "content": "How much is parking at the venue?"}],
        "answer": "On-site parking is $40 per vehicle.",
        "key_points": ["$40"],
    },
    {
        "id": "qa_007",
        "messages": [{"role": "user", "content": "Do children under 2 need a ticket?"}],
        "answer": "Children under 2 years old do not need a ticket.",
        "key_points": ["under 2", "do not need a ticket"],
    },
    {
        "id": "qa_008",
        "messages": [{"role": "user", "content": "What's the refund policy if I can't attend?"}],
        "answer": "Full refunds are available up to 24 hours before kickoff.",
        "key_points": ["full refund", "24 hours"],
    },
]

In [7]:
# TODO: perform the evaluation

# References

Here are the references I used to build this ressource.

- [Beginner's Guide to Agent Evaluations - Langchain on YouTube](https://www.youtube.com/watch?v=_QozKR9eQE8)

- [Agents Course - HuggingFace](https://huggingface.co/learn/agents-course/unit0/introduction)